# 05 -- Flow Matching & Rectified Flow

**Goal:** learn a *velocity field* that carries Gaussian noise to data along (nearly) straight paths,
then generate by solving an ODE. Simpler than diffusion, fewer sampling steps, and the training objective
behind Stable Diffusion 3 and FLUX.

**In this notebook:** probability paths and the conditional flow-matching loss, a 2-D toy you can train in
seconds, the image model, ODE solvers and step counts, and Hub upload.

*Papers:* Lipman et al., *Flow Matching for Generative Modeling* (2022) · Liu et al., *Rectified Flow* (2022) · Albergo & Vanden-Eijnden (2022) · Esser et al., *SD3* (2024).

## 1. Theory

### A path from noise to data
Pick noise $x_0\sim\mathcal N(0,I)$ and a data point $x_1$. Join them with a straight line:
$$
x_t=(1-t)\,x_0+t\,x_1,\qquad t\in[0,1]
$$
(convention here: $t=0$ noise, $t=1$ data -- the reverse of diffusion's $t$). Along this line the velocity is constant:
$u = \frac{dx_t}{dt} = x_1-x_0$.

### The conditional flow-matching loss
We want a single field $v_\theta(x,t)$ that moves **the whole noise distribution** onto the data distribution.
The marginal velocity at a point $x_t$ is the average of $x_1-x_0$ over all pairs whose line passes through $x_t$ --
intractable directly, but Lipman et al. show that regressing the *per-pair* target has the same gradient:
$$
\mathcal L_\text{CFM}(\theta)=\mathbb E_{t,\,x_0,\,x_1}\big\|v_\theta(x_t,t)-(x_1-x_0)\big\|^2 .
$$
That is the entire training algorithm: sample noise, sample data, sample $t$, regress. No noise schedule, no $\bar\alpha_t$,
no variance terms. (`sigma_min>0` gives the original FM path with a small terminal noise; 0 is rectified flow.)

### Sampling = solving an ODE
$\frac{dx}{dt}=v_\theta(x,t)$ from $x(0)\sim\mathcal N(0,I)$ to $t=1$.
* **Euler**: $x\leftarrow x+\Delta t\,v_\theta(x,t)$ -- one network call per step.
* **Heun**: Euler predictor + trapezoidal corrector -- two calls per step, second-order accurate.

Lines for different pairs cross, so the learned *marginal* paths are curved -- but much straighter than diffusion's,
which is why flow matching needs fewer steps. **Reflow** (retraining on the model's own (noise, sample) pairs)
straightens them further, toward one-step generation.

### Relation to diffusion
Diffusion with a Gaussian path is a special case of flow matching with a different (curved) interpolation; the
noise-prediction network and the velocity network are linear transforms of each other. Same U-Net, same
classifier-free guidance -- only the path and the loss target change.

### Time sampling
Uniform $t$ wastes effort near the endpoints where the target is easy. `time_sampling: logit_normal`
($t=\sigma(n),\,n\sim\mathcal N(0,1)$) concentrates training in the middle, as in SD3.

## 0. Setup

Works in three places, no edits needed:

* **Google Colab** -- the cell below downloads the code from GitHub and installs it (~1 min).
  Choose a GPU first: *Runtime -> Change runtime type -> T4 GPU*.
* **RunPod** -- run `bash scripts/runpod/setup.sh` once, then open JupyterLab (port 8888).
* **Your own machine** -- `pip install -e ".[notebook,finetune]"` in the repo, then open the notebook.

In [ ]:
import sys, pathlib, subprocess

# Where is the code? If this notebook was opened on its own (e.g. in Colab), fetch it first.
REPO_URL = "https://github.com/yar78/vision-generative-models-survey.git"
BRANCH = "claude/vibrant-noether-ftcsah"   # change to "main" once this branch is merged

REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "genmodels").is_dir():
    REPO = pathlib.Path("vision-generative-models-survey").resolve()
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}"], check=True)
sys.path.insert(0, str(REPO))

import torch
import yaml
from genmodels import build_trainer
from genmodels.common.config import load_config
from genmodels.common.data import get_dataset
from genmodels.common.utils import get_device, load_images, nearest_neighbours, plot_history, show_grid
from genmodels.hub import push_run, resolve_repo_id
from genmodels.flow_matching import FlowMatchingModel
print("code:", REPO)
print("device:", get_device(), f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "-- no GPU: training will be slow")

### A 2-D toy first

Before images: learn to move a 2-D Gaussian onto a two-moons distribution with a tiny MLP. It trains in seconds
on a CPU and shows everything -- the loss, the learned field, and the trajectories that the ODE solver follows.

In [ ]:
import math
import matplotlib.pyplot as plt
from torch import nn

def two_moons(n):
    t = torch.rand(n) * math.pi
    a = torch.stack([torch.cos(t), torch.sin(t)], 1)
    b = torch.stack([1 - torch.cos(t), 0.5 - torch.sin(t)], 1)
    x = torch.where(torch.rand(n, 1) < 0.5, a, b)
    return (x - torch.tensor([0.5, 0.25])) * 1.5 + 0.05 * torch.randn(n, 2)

toy = nn.Sequential(nn.Linear(3, 128), nn.SiLU(), nn.Linear(128, 128), nn.SiLU(), nn.Linear(128, 2))
opt = torch.optim.Adam(toy.parameters(), 1e-3)
for step in range(3000):
    x1 = two_moons(512)
    x0 = torch.randn_like(x1)
    t = torch.rand(len(x1), 1)
    xt = (1 - t) * x0 + t * x1                       # the straight path
    loss = ((toy(torch.cat([xt, t], 1)) - (x1 - x0)) ** 2).mean()   # regress the velocity
    opt.zero_grad(); loss.backward(); opt.step()
print("final loss", loss.item())

@torch.no_grad()
def toy_sample(n, steps):
    x = torch.randn(n, 2); path = [x]
    for i in range(steps):
        t = torch.full((n, 1), i / steps)
        x = x + toy(torch.cat([x, t], 1)) / steps
        path.append(x)
    return torch.stack(path)

path = toy_sample(1000, 50)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].scatter(*path[0].T, s=2); ax[0].set_title("t=0: noise")
for i in range(0, 1000, 10):
    ax[1].plot(*path[:, i].T, lw=0.5, alpha=0.6)
ax[1].set_title("ODE trajectories")
ax[2].scatter(*path[-1].T, s=2, c="C1"); ax[2].set_title("t=1: samples")
for a in ax: a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_aspect("equal")
plt.show()

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, steps in zip(ax, (1, 2, 4, 50)):
    s = toy_sample(1000, steps)[-1]
    a.scatter(*s.T, s=2); a.set_title(f"{steps} Euler step(s)"); a.set_xlim(-3, 3); a.set_ylim(-3, 3); a.set_aspect("equal")
plt.show()

## 2. Choose an experiment

Every experiment is one YAML file in `configs/`, shared with the command-line script,
so a run started here and a run started on a RunPod GPU are the same experiment.

* `configs/flow_matching/mnist.yaml` -- the quick default.
* `configs/ray_dog/flow_matching.yaml` -- your **Ray_dog** photos (`yaryar78/Ray_dog`, 258 images) at 64x64.
* `configs/flow_matching/cifar10.yaml` -- CIFAR-10, matched to `configs/diffusion/ddpm_cifar10.yaml`.

Edit `cfg` below to change anything (steps, batch size, model width ...).

In [ ]:
CONFIG = REPO / "configs/flow_matching/mnist.yaml"        # or: REPO / "configs/ray_dog/flow_matching.yaml"
cfg = load_config(CONFIG)

# Override anything here, for example:
# cfg["train"]["steps"] = 5000

print(yaml.safe_dump(cfg, sort_keys=False))

### Look at the training data (with the training-time augmentation applied)

In [ ]:
train_set = get_dataset(dict(cfg["data"], num_workers=0))
print(f"{len(train_set)} training images")
x_real = torch.stack([train_set[i][0] for i in torch.randperm(len(train_set))[:64].tolist()])
show_grid(x_real, title="training data")

### The probability path on real images
$x_t=(1-t)x_0+tx_1$ for $t=0,0.1,\dots,1$: a straight blend from noise to image -- compare with the diffusion forward process.

In [ ]:
x1 = x_real[:3]
x0 = torch.randn_like(x1)
ts = torch.linspace(0, 1, 11)
show_grid(torch.stack([(1 - t) * x0 + t * x1 for t in ts], 1).flatten(0, 1), nrow=11, title="x_t, t = 0 ... 1", size=11)

## 3. Build the model

`build_trainer(cfg)` creates the U-Net velocity field, the optimizers and the training loop.
Outputs go to `outputs/<name>/` (on RunPod: `/workspace/genmodels/outputs/<name>/`, on the network volume).

In [ ]:
trainer = build_trainer(cfg)
print("run dir:", trainer.run_dir)
print({k: f"{v/1e6:.2f}M params" for k, v in trainer.num_params().items()})

### The training objective, as implemented

In [ ]:
import inspect; print(inspect.getsource(FlowMatchingModel.loss))

## 4. Train

`trainer.fit()` runs `train.steps` steps. Every `sample_every` steps it saves a fixed-seed
sample grid to `samples/`; every `ckpt_every` steps a checkpoint. Re-running the cell
(or the script) **resumes** from the checkpoint.

MNIST config: ~20 min on a GPU (same budget as the DDPM notebook). Ray_dog: ~2-3 h for 40k steps.

Already trained this config with the script (e.g. on RunPod)? Set `TRAIN = False`
and the checkpoint in the run directory is loaded instead.

### Start TensorBoard first, then train

Run this cell **before** the training cell. The dashboard updates live while training runs
(it refreshes every 30 s; the reload button at its top right forces it):

* **Scalars** -- the loss curves.
* **Images** -- the fixed-seed sample grid, with a slider over training steps.

On RunPod with the script, TensorBoard is also served on port 6006 by `scripts/runpod/train.sh`.

Started training without it? Stop the training cell (■), run this one, then run the training
cell again: it **continues from the same step** -- the model and optimizer are still in memory.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {trainer.run_dir}/tensorboard

In [ ]:
TRAIN = True

if TRAIN:
    trainer.fit()
elif trainer.ckpt_path.exists():
    trainer.load_checkpoint()
    print(f"loaded checkpoint at step {trainer.step}")
else:
    print("No checkpoint yet in", trainer.run_dir)

### Loss curves
The same curves as TensorBoard's Scalars tab, drawn here once training has stopped.

In [ ]:
plot_history(trainer.history)

## 5. Results
### Samples (EMA weights)

In [ ]:
show_grid(trainer.sample_fixed(64), title='samples')

## 6. Inside the sampler
### Trajectory: the image at intermediate $t$ of the ODE

In [ ]:
model = trainer.net_for_sampling("model").eval()
y = torch.arange(4, device=trainer.device) % model.num_classes if model.num_classes else None
torch.manual_seed(0)
_, traj = model.sample(4, steps=20, y=y, guidance_scale=trainer.guidance, device=trainer.device, return_trajectory=True)
picks = traj[::2]
show_grid(torch.stack(picks, 1).flatten(0, 1).clamp(-1, 1), nrow=len(picks), title="x_t along the ODE, t = 0 ... 1", size=12)

### Steps and solvers
Same starting noise for every row. Rows: Euler with 1, 2, 4, 8, 20 steps, then Heun with 4 and 10 steps (8 and 20 network calls).

In [ ]:
n = 8
y = (torch.arange(n, device=trainer.device) % model.num_classes) if model.num_classes else None
x0 = torch.randn(n, model.in_channels, model.image_size, model.image_size, device=trainer.device)
rows, labels = [], []
for solver, steps in [("euler", 1), ("euler", 2), ("euler", 4), ("euler", 8), ("euler", 20), ("heun", 4), ("heun", 10)]:
    rows.append(model.sample(n, steps=steps, solver=solver, y=y, guidance_scale=trainer.guidance, device=trainer.device, x0=x0))
    labels.append(f"{solver}-{steps}")
show_grid(torch.cat(rows), nrow=n, title=" / ".join(labels), size=9)

### Classifier-free guidance (conditional configs only)

In [ ]:
if model.num_classes:
    y = torch.arange(model.num_classes, device=trainer.device)
    x0 = torch.randn(len(y), model.in_channels, model.image_size, model.image_size, device=trainer.device)
    rows = [model.sample(len(y), steps=20, y=y, guidance_scale=w, device=trainer.device, x0=x0) for w in (0.0, 1.0, 2.0, 4.0)]
    show_grid(torch.cat(rows), nrow=len(y), title="guidance w = 0, 1, 2, 4 (top to bottom)", size=9)
else:
    print("Unconditional model -- set method.conditional: true to try guidance.")

### Memorisation check

With a small dataset (Ray_dog has 258 images) a generative model can simply copy training
images. For each sample (top row) we show its nearest training image in pixel space (bottom
row). Near-identical pairs mean memorisation; similar-but-different means the model generalises.

In [ ]:
eval_set = get_dataset(dict(cfg["data"], random_crop=False, hflip=False, num_workers=0))
train_imgs = load_images(eval_set, max_images=10_000)
samples = trainer.sample_fixed(8).cpu()
nn_imgs = nearest_neighbours(samples, train_imgs)[:, 0]
show_grid(torch.cat([samples, nn_imgs]), nrow=8, title="samples (top) vs nearest training image (bottom)", size=10)

## 7. Push to the Hugging Face Hub

`push_run` uploads the run directory as a model repo **`<your-username>/{cfg['name']}`**
(private by default):

| file | |
|---|---|
| `model.safetensors`, `config.json` | weights + constructor args (`FlowMatchingModel.from_pretrained` reads these) |
| `README.md` | generated model card: dataset, steps, hardware, git commit, metrics |
| `samples.png`, `metrics.json`, `train_config.yaml` | results and the exact config |
| `runs/` | TensorBoard logs (shown in the Hub's *Training metrics* tab) |

Login once with a **write** token from https://huggingface.co/settings/tokens
(`notebook_login()` below, `huggingface-cli login`, or the `HF_TOKEN` env var). To use another
repo name set `cfg["hub"]["repo_id"]`; to publish, `cfg["hub"]["private"] = False` before training,
or `push_run(..., private=False)`.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, whoami
    try:
        print("logged in as", whoami()["name"])
    except Exception:
        notebook_login()
    url = push_run(trainer.run_dir)

### Reload it from the Hub
This is how anyone (including future you) uses the model.

In [ ]:
if PUSH_TO_HUB:
    from genmodels.flow_matching import FlowMatchingModel
    model = FlowMatchingModel.from_pretrained(resolve_repo_id(cfg)).to(trainer.device).eval()
    show_grid(model.generate(16, steps=20), title="reloaded from the Hub")

## 8. Things to try

* Train the MNIST configs of this notebook and `04_diffusion` for the same steps, then compare quality at 5 / 10 / 20 sampling steps.
* `method.time_sampling: logit_normal` vs `uniform`.
* **Reflow:** sample (noise, image) pairs from the trained model, retrain on those pairs, and check how many Euler steps you now need.